# DSCI 552 — Homework 3

**Name:** Long Zhang  
**GitHub Username:** **[zl535992549-cell]**  
**USC ID:** **8768388438**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_SEED = 552
DATA_DIR = Path("../data/AReM")

## Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download and inspect the AReM dataset


In [ ]:
COLUMN_NAMES = [
    "time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23",]
SENSOR_COLUMNS = COLUMN_NAMES[1:]
EXPECTED_TIMES = pd.Index(np.arange(0, 120000, 250), name="time")
cleaning_log = []


def load_sensor_file(path):
    parsed = []
    with path.open("r", encoding="utf-8-sig") as source:
        for line_number, raw_line in enumerate(source, start=1):
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue

            fields = line.replace(",", " ").split()
            if len(fields) != 7:
                raise ValueError(f"{path}: line {line_number} has {len(fields)} fields, expected 7")
            parsed.append([float(value) for value in fields])

    frame = pd.DataFrame(parsed, columns=COLUMN_NAMES).set_index("time").sort_index()
    if frame.index.has_duplicates:
        raise ValueError(f"Duplicate timestamp in {path}")
    extra_times = frame.index.difference(EXPECTED_TIMES)
    if len(extra_times) > 0:
        raise ValueError(f"Unexpected timestamps in {path}: {list(extra_times)}")

    missing_times = EXPECTED_TIMES.difference(frame.index)
    if len(missing_times) > 0:
        frame = frame.reindex(EXPECTED_TIMES).interpolate(method="index", limit_direction="both")
        cleaning_log.append({
            "File": str(path.relative_to(DATA_DIR)),
            "Missing timestamps filled (ms)": list(missing_times.astype(int)),
            "Method": "Linear interpolation",})

    if frame.shape != (480, 6) or frame.isna().any().any():
        raise ValueError(f"Invalid processed data shape / missing values: {path}: {frame.shape}")
    return frame

assert DATA_DIR.is_dir(), "Expected the dataset at ../data/AReM"

Validated 88 CSV instances with 480 time points and 6 channels each.
Missing-time repairs:


,File,Missing timestamps filled (ms),Method
0,sitting/dataset8.csv,[13500],Linear interpolation


Other tolerated source irregularities: bending2/dataset4 uses spaces; cycling/dataset9 and dataset14 have a trailing comma on the last row.


### (b) Training and test datasets

In [ ]:
ACTIVITIES = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
records = []
for activity in ACTIVITIES:
    files = sorted(
        (DATA_DIR / activity).glob("dataset*.csv"),
        key=lambda file: int(file.stem.removeprefix("dataset")),)
    for file in files:
        dataset_number = int(file.stem.removeprefix("dataset"))
        test_limit = 2 if activity.startswith("bending") else 3
        records.append({
            "Activity": activity,
            "Dataset": file.name,
            "Split": "Test" if dataset_number <= test_limit else "Train",
            "Path": file,})

catalogue = pd.DataFrame(records)
split_counts = pd.crosstab(catalogue["Activity"], catalogue["Split"])
display(split_counts)

train_files = catalogue.loc[catalogue["Split"] == "Train"].copy()
test_files = catalogue.loc[catalogue["Split"] == "Test"].copy()
assert len(catalogue) == 88
assert len(train_files) == 69 and len(test_files) == 19
assert catalogue["Path"].is_unique
print(f"Training instances: {len(train_files)}; Test instances: {len(test_files)}; Total: {len(catalogue)}")

Split,Test,Train
Activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


Training instances: 69; Test instances: 19; Total: 88


### (c) Feature Extraction

#### (i) Common Time-Domain Features

Common time-domain features include range, interquartile range, RMS, minimum, mean, median, maximum, quartiles, standard deviation, variance, skewness, autocorrelation, and kurtosis.


#### (ii) Extract Seven Time-Domain Features
42 features per instance(88 × 42)

In [ ]:
FEATURE_TYPES = ["min", "max", "mean", "median", "std", "1st quart", "3rd quart"]

def extract_time_features(frame):
    extracted = {}
    for channel_number, sensor_name in enumerate(SENSOR_COLUMNS, start=1):
        x = frame[sensor_name]
        extracted[f"min{channel_number}"] = x.min()
        extracted[f"max{channel_number}"] = x.max()
        extracted[f"mean{channel_number}"] = x.mean()
        extracted[f"median{channel_number}"] = x.median()
        extracted[f"std{channel_number}"] = x.std(ddof=1)
        extracted[f"1st quart{channel_number}"] = x.quantile(0.25)
        extracted[f"3rd quart{channel_number}"] = x.quantile(0.75)
    return extracted


feature_rows = []
for instance_number, entry in enumerate(catalogue.itertuples(index=False), start=1):
    sensor_frame = load_sensor_file(entry.Path)
    feature_rows.append({
        "Instance": instance_number,
        "Activity": entry.Activity,
        "Dataset": entry.Dataset,
        "Split": entry.Split,
        **extract_time_features(sensor_frame),})

features = pd.DataFrame(feature_rows)
feature_columns = [
    f"{feature_name}{channel_number}"
    for channel_number in range(1, 7)
    for feature_name in FEATURE_TYPES]
assert features.shape == (88, 46)
assert features.loc[:, feature_columns].shape == (88, 42)
assert features[feature_columns].notna().all().all()

print("Full feature table:", features.shape, "| Numeric feature matrix: (88, 42)")
display(features.head(8).round(3))
print("Example, first instance (first sensor channel):")
display(features.loc[:0, ["Instance", "min1", "max1", "mean1", "median1", "std1", "1st quart1", "3rd quart1"]].round(3))

Full feature table: (88, 46) | Numeric feature matrix: (88, 42)


,Instance,Activity,Dataset,Split,min1,max1,...,max6,mean6,median6,std6,1st quart6,3rd quart6
0,1,bending1,dataset1.csv,Test,37.25,45.00,...,1.92,0.571,0.43,0.583,0.00,1.30
1,2,bending1,dataset2.csv,Test,38.00,45.67,...,3.11,0.571,0.43,0.601,0.00,1.30
2,3,bending1,dataset3.csv,Train,35.00,47.40,...,1.79,0.493,0.43,0.514,0.00,0.94
3,4,bending1,dataset4.csv,Train,33.00,47.75,...,2.18,0.614,0.50,0.524,0.00,1.00
4,5,bending1,dataset5.csv,Train,33.00,45.75,...,1.79,0.383,0.43,0.389,0.00,0.50
5,6,bending1,dataset6.csv,Train,37.00,48.00,...,5.26,0.680,0.50,0.623,0.43,0.87
6,7,bending1,dataset7.csv,Train,36.25,48.00,...,2.96,0.555,0.49,0.488,0.00,0.83
7,8,bending2,dataset1.csv,Test,12.75,51.00,...,4.97,0.700,0.50,0.694,0.43,0.87


Example, first instance (first sensor channel):


,Instance,min1,max1,mean1,median1,std1,1st quart1,3rd quart1
0,1,37.25,45.0,40.625,40.5,1.477,39.25,42.0


In [ ]:
feature_output = RESULTS_DIR / "AReM_88_instances_42_features.csv"
features.to_csv(feature_output, index=False, float_format="%.6f")
print(f"Saved the complete 88-instance, 42-feature table to {feature_output.as_posix()}")

Saved the complete 88-instance, 42-feature table to ../results/AReM_88_instances_42_features.csv


#### (iii) Standard Deviation and 90% Bootstrap Confidence Intervals

In [ ]:
feature_values = features.loc[:, feature_columns].to_numpy(dtype=float)
point_sd = np.std(feature_values, axis=0, ddof=1)

n_resamples = 10_000
batch_size = 500
rng = np.random.default_rng(RANDOM_SEED)
boot_sd = np.empty((n_resamples, feature_values.shape[1]), dtype=float)

for start in range(0, n_resamples, batch_size):
    stop = min(start + batch_size, n_resamples)
    indices = rng.integers(0, len(feature_values), size=(stop - start, len(feature_values)))
    resampled_feature_tables = feature_values[indices]
    boot_sd[start:stop, :] = resampled_feature_tables.std(axis=1, ddof=1)

low, high = np.quantile(boot_sd, [0.05, 0.95], axis=0)
bootstrap_summary = pd.DataFrame({
    "Feature": feature_columns,
    "Estimated_SD": point_sd,
    "CI_90_Lower": low,
    "CI_90_Upper": high,
})

assert len(bootstrap_summary) == 42
assert np.isfinite(boot_sd).all()
assert (bootstrap_summary["CI_90_Lower"] <= bootstrap_summary["CI_90_Upper"]).all()
assert (bootstrap_summary["Estimated_SD"] >= 0).all()
print("Sample standard deviations and their 90% bootstrap percentile intervals (all 42 features):")
display(bootstrap_summary.round(3))

Sample standard deviations and their 90% bootstrap percentile intervals (all 42 features):


,Feature,Estimated_SD,CI_90_Lower,CI_90_Upper
0,min1,9.570,8.262,10.761
1,max1,4.394,3.328,5.285
2,mean1,5.336,4.704,5.886
3,median1,5.440,4.790,6.004
4,std1,1.772,1.572,1.950
5,1st quart1,6.154,5.582,6.652
6,3rd quart1,5.139,4.332,5.841
7,min2,0.000,0.000,0.000
8,max2,5.063,4.623,5.402
9,mean2,1.574,1.394,1.707


In [ ]:
bootstrap_output = RESULTS_DIR / "AReM_feature_standard_deviations_90pct_CI.csv"
bootstrap_summary.to_csv(bootstrap_output, index=False, float_format="%.6f")
print(f"Saved all 42 bootstrap results to {bootstrap_output.as_posix()}")

Saved all 42 bootstrap results to ../results/AReM_feature_standard_deviations_90pct_CI.csv


#### (iv) Three Selected Time-Domain Features

**Mean, standard deviation, and maximum** — representing typical level, variability, and peak intensity.

## 2. ISLR Chapter 3, Exercise 3.7.4

**(a)** Cubic regression has lower or equal training RSS because it is more flexible.

**(b)** Linear regression is expected to have lower test RSS because the true relationship is linear.

**(c)** Cubic regression has lower or equal training RSS because it is more flexible.

**(d)** Cannot determine because test RSS depends on the degree of nonlinearity and overfitting.

References

* UCI AReM dataset
* ChatGPT
* Time-series features
* An Introduction to Statistical Learning, Chapter 3, Exercise 4.